# 🔮 04 — Prophet avec COVID comme Holiday
## Modèle Prophet avec grid search des hyperparamètres

Ce notebook utilise Prophet en déclarant la période COVID comme holiday. 3 configurations testées avec différents `holidays_prior_scale`.

## 1. 📦 Imports & Chargement

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("✅ Imports OK")

In [ ]:
# Charger les données (Prophet utilise ds/y)
df = pd.read_csv("prepared_data.csv", index_col=0, parse_dates=True).reset_index()
df = df.rename(columns={"date": "ds", "revenue": "y"})

prophet_train = df[df["ds"].dt.year < 2022].copy()
prophet_test = df[df["ds"].dt.year == 2022].copy()

print(f"✅ Données chargées")
print(f"   Train : {len(prophet_train)} mois → Test : {len(prophet_test)} mois")

## 2. 🦠 Définition des Holidays COVID

In [ ]:
covid_events = []
# COVID sévère
for y, m in [(2020,3),(2020,4),(2020,5),(2020,6)]:
    covid_events.append({"holiday": "covid_severe", "ds": pd.Timestamp(f"{y}-{m:02d}-01"),
                         "lower_window": 0, "upper_window": 0})
# COVID modéré
for y, m in [(2020,7),(2020,8),(2020,9),(2020,10),(2020,11),(2020,12),
             (2021,1),(2021,2),(2021,3),(2021,4),(2021,5),(2021,6)]:
    covid_events.append({"holiday": "covid_moderate", "ds": pd.Timestamp(f"{y}-{m:02d}-01"),
                         "lower_window": 0, "upper_window": 0})
# Noël
for y in range(2018, 2024):
    covid_events.append({"holiday": "christmas", "ds": pd.Timestamp(f"{y}-12-01"),
                         "lower_window": 0, "upper_window": 0})

holidays_df = pd.DataFrame(covid_events)
print(f"✅ {len(holidays_df)} holidays définis (sévère={4}, modéré={12}, noël={6})")
print("   ⚠️ upper_window=0 (pas de fenêtre, données mensuelles)")

## 3. ⚙️ Grid Search Prophet (3 configurations)

In [ ]:
prophet_configs = [
    {"name": "Prophet (covid_prior=3)", "changepoint": 0.02, "seasonality": 5.0, "holidays": 3.0},
    {"name": "Prophet (covid_prior=5)", "changepoint": 0.03, "seasonality": 8.0, "holidays": 5.0},
    {"name": "Prophet (covid_prior=8)", "changepoint": 0.05, "seasonality": 10.0, "holidays": 8.0},
]

best_prophet, best_mape, best_name = None, np.inf, ""
best_forecast, best_pred = None, None

print("⏳ Grid Search Prophet...")
for cfg in prophet_configs:
    model = Prophet(holidays=holidays_df, yearly_seasonality=True,
                    weekly_seasonality=False, daily_seasonality=False,
                    seasonality_mode="multiplicative",
                    changepoint_prior_scale=cfg["changepoint"],
                    seasonality_prior_scale=cfg["seasonality"],
                    holidays_prior_scale=cfg["holidays"],
                    interval_width=0.95)
    model.add_seasonality(name="monthly", period=30.5, fourier_order=3)
    model.fit(prophet_train)
    
    future = model.make_future_dataframe(periods=12, freq="MS")
    forecast = model.predict(future)
    
    forecast_test = forecast[forecast["ds"].dt.year == 2022].copy()
    y_pred = forecast_test.set_index("ds")["yhat"].clip(lower=0)
    y_true = prophet_test.set_index("ds")["y"]
    
    common_idx = y_true.index.intersection(y_pred.index)
    mape_val = np.mean(np.abs((y_true[common_idx].values - y_pred[common_idx].values) / y_true[common_idx].values)) * 100
    
    print(f"   {cfg['name']:<30} → MAPE: {mape_val:.2f}%")
    
    if mape_val < best_mape:
        best_mape = mape_val
        best_prophet = model
        best_name = cfg["name"]
        best_forecast = forecast
        best_pred = y_pred[common_idx]

y_test_prophet = prophet_test.set_index("ds")["y"]
common_idx = y_test_prophet.index.intersection(best_pred.index)

mae = mean_absolute_error(y_test_prophet[common_idx], best_pred)
rmse = np.sqrt(mean_squared_error(y_test_prophet[common_idx], best_pred))
r2 = r2_score(y_test_prophet[common_idx], best_pred)

print(f"\n✅ Meilleur : {best_name}")
print(f"   MAE  : {mae:>12,.0f} €")
print(f"   RMSE : {rmse:>12,.0f} €")
print(f"   MAPE : {best_mape:>11.2f} %")
print(f"   R²   : {r2:>11.4f}")

## 4. 📊 Visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 11))

ax = axes[0]
prophet_train.set_index("ds")["y"].plot(ax=ax, label="Train", color="#1565C0", alpha=0.7, linewidth=2)
y_test_prophet.plot(ax=ax, label="Test réel", color="#2E7D32", linewidth=2.5)
best_pred.plot(ax=ax, label="Prophet forecast", color="#7B1FA2", linestyle="--", linewidth=2.5)

forecast_test = best_forecast[best_forecast["ds"].dt.year == 2022].set_index("ds")
ax.fill_between(forecast_test.index, forecast_test["yhat_lower"].clip(lower=0),
                forecast_test["yhat_upper"], alpha=0.15, color="#7B1FA2", label="IC 95%")
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-30"), alpha=0.07, color="red", label="COVID")
ax.set_title("Prophet — Prévisions vs Réel", fontsize=13, fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M€"))
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
trend_full = best_forecast.set_index("ds")["trend"]
trend_full.plot(ax=ax2, color="#E65100", linewidth=2.5, label="Tendance")
ax2.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-30"), alpha=0.07, color="red")
ax2.set_title("Tendance extraite par Prophet", fontsize=12, fontweight="bold")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M€"))
ax2.grid(alpha=0.3); ax2.legend()

plt.suptitle(f"Prophet | MAPE: {best_mape:.2f}% | R²: {r2:.4f}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("modele4_prophet.png", dpi=120, bbox_inches="tight")
plt.show()
print("💾 Graphique → modele4_prophet.png")

# Composantes
fig2 = best_prophet.plot_components(best_forecast)
fig2.suptitle("Prophet — Décomposition", fontsize=13, fontweight="bold")
plt.tight_layout()
fig2.savefig("modele4_prophet_components.png", dpi=120, bbox_inches="tight")
plt.show()
print("💾 Composantes → modele4_prophet_components.png")

# Sauvegarder
results_df = pd.DataFrame({"date": y_test_prophet.index, "reel": y_test_prophet, "prediction": best_pred})
results_df.to_csv("predictions_prophet.csv", index=False)
print("💾 Prédictions → predictions_prophet.csv")